# SECCIÓN 1 — Introducción
En esta fase se realiza la integración de las tres tablas principales del M5 Forecasting: ventas, calendario y precios.<br>
El objetivo es transformar el dataset desde su formato original (ventas en formato wide) hacia un formato unificado y estructurado que permita la posterior generación de variables supervisadas (lags, rolling windows, codificación categórica).<br>
Esta etapa incluye:<br>

Conversión de ventas a formato `long`<br>

Unión con el calendario mediante la clave `d`<br>

Unión con precios mediante `store_id`, `item_id` y `wm_yr_wk`<br>

Validaciones de integridad tras el merge<br>

Revisión de duplicados y consistencia temporal<br>

El resultado final será un dataset limpio y unificado, almacenado en data/processed/m5_clean.parquet, listo para la fase de Feature Engineering.

# SECCIÓN 2 — Configuración de rutas y carga de datos

In [1]:
import os
import sys

# Obtener ruta actual
current_path = os.getcwd()

# Subir dos niveles: notebooks → M3
PROJECT_ROOT = os.path.abspath(
    os.path.join(current_path, "..", "..") #porque estamos en la carpeta exploratory/ - de pruebas
)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

In [2]:
sales, calendar, prices = load_m5_data()

print("Sales:", sales.shape)
print("Calendar:", calendar.shape)
print("Prices:", prices.shape)

Sales: (30490, 1919)
Calendar: (1969, 14)
Prices: (6841121, 4)


# SECCIÓN 3 — Conversión de formato wide a formato long
## Convertir ventas a formato long

In [3]:
id_vars = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
value_vars = [c for c in sales.columns if c.startswith("d_")]

sales_long = sales.melt(
    id_vars=id_vars,
    value_vars=value_vars,
    var_name="d",
    value_name="sales"
)

sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


In [4]:
# Validación del long format
print("Filas en formato long:", sales_long.shape)
assert sales_long["id"].nunique() == sales.shape[0]
assert sales_long["d"].nunique() == len(value_vars)

Filas en formato long: (58327370, 8)


# SECCIÓN 3.1 — Evaluación del impacto computacional
Tras la conversión del dataset al formato long, el número de registros aumenta hasta aproximadamente 58 millones de filas.<br>
Este incremento significativo en el tamaño del dataset implica un mayor consumo de memoria y tiempo de procesamiento, especialmente durante operaciones posteriores como merges y generación de variables temporales.<br>
Durante las pruebas iniciales en pandas, estas operaciones generaron tiempos de ejecución elevados y consumo intensivo de memoria, lo que motivó la evaluación de soluciones distribuidas.<br>
Este comportamiento constituye uno de los principales motivos para la posterior migración del pipeline a Apache Spark.

# SECCIÓN 4 — Merge con calendario

In [5]:
df = sales_long.merge(calendar, on="d", how="left")

In [6]:
# Validación del merge
assert df["date"].notna().all(), "Error: hay días sin correspondencia en calendar"

La unión con el calendario añade información temporal esencial (fecha, mes, año, eventos, SNAP), necesaria para capturar patrones estacionales y efectos de calendario.

# SECCIÓN 5 — Merge con precios

In [7]:
df = df.merge(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")

In [8]:
# Validación del merge
missing_prices = df["sell_price"].isna().mean()
missing_prices

np.float64(0.2108686367994991)

# SECCIÓN 5.1 — Verificar duplicados tras merge

In [ ]:
# Validación de duplicados tras merge
duplicates = df.duplicated(
    subset=["id", "d"]
).sum()

print("Duplicados tras merge:", duplicates)

assert duplicates == 0, "Error: duplicados detectados tras merge"

# SECCIÓN 5.2 — Verificar valores faltantes tras merge

In [9]:
df[df["sell_price"].isna()].groupby("item_id").size().sort_values(ascending=False).head(10)


item_id
FOODS_2_379        16030
FOODS_3_296        15596
HOUSEHOLD_1_242    15540
HOUSEHOLD_1_159    15379
HOUSEHOLD_1_308    15246
FOODS_2_253        15155
HOUSEHOLD_2_186    15078
FOODS_3_353        15036
HOUSEHOLD_1_489    14889
HOUSEHOLD_1_484    14882
dtype: int64

# SECCIÓN 5.3 — Análisis de valores faltantes en sell_price tras merge
Tras el merge con la tabla de precios, se observa un 21.08% de valores faltantes en la variable sell_price.
Este comportamiento es esperado en el dataset M5, ya que la tabla sell_prices.csv únicamente registra precios cuando existe una variación semanal.<br>
Como consecuencia, muchos productos mantienen un precio constante durante largos periodos y no aparecen en la tabla de precios para todas las semanas.<br>
El análisis muestra que los valores faltantes se concentran en un subconjunto reducido de productos (principalmente categorías FOODS_2, FOODS_3 y HOUSEHOLD_1), lo cual confirma que no se trata de un error en el proceso de merge.
Estos valores faltantes no afectan al proceso de modelado, ya que los modelos basados en árboles manejan nulos de forma nativa y la variable sell_price no interviene en la métrica WRMSSE ni en la reconciliación jerárquica.
Por tanto, no es necesaria ninguna imputación en esta fase.

# SECCIÓN 5.4 — Tratamiento de valores faltantes
Para tratar los valores faltantes en sell_price, se aplica una imputación basada en continuidad temporal mediante forward-fill y backward-fill dentro de cada combinación store_id–item_id.<br>
Este enfoque preserva la coherencia temporal del precio y evita la introducción de valores artificiales. La ausencia de precio en determinadas semanas se interpreta como continuidad del último precio conocido.

In [10]:
df["sell_price"] = df.groupby(["store_id","item_id"])["sell_price"].ffill()
df["sell_price"] = df.groupby(["store_id","item_id"])["sell_price"].bfill()

In [11]:
# Volvemos a validar el merge
missing_prices = df["sell_price"].isna().mean()
missing_prices

np.float64(0.0)

# SECCIÓN 5.5 — Validación temporal final tras merge

In [ ]:
df["date"].min(), df["date"].max()

Se verifica que el rango temporal del dataset integrado coincide con el calendario original, garantizando la continuidad temporal necesaria para la generación posterior de variables supervisadas.

# SECCIÓN 6 — Guardar el dataset M5 limpio
El almacenamiento en formato Parquet permite una compresión eficiente y un acceso columnar optimizado, facilitando la posterior lectura por motores distribuidos como Apache Spark.

In [12]:
# Guardar dataset limpio
os.makedirs("data/processed/exploratory", exist_ok=True)
df.to_parquet("data/processed/exploratory/m5_clean.parquet", index=False)

print("Dataset limpio guardado.")


Dataset limpio guardado.


# SECCIÓN 6.1 — Guardar sample del dataset M5 limpio para pruebas de features engineering
Se genera un subconjunto reducido del dataset compuesto por 50 series temporales seleccionadas aleatoriamente. Este subconjunto permite realizar pruebas rápidas y validar la lógica de transformación sin necesidad de procesar el dataset completo en cada iteración.

In [13]:
# Tomar 50 series aleatorias para pruebas y poder avanzar rápido
ids = df["id"].drop_duplicates().sample(50, random_state=42)
df_small = df[df["id"].isin(ids)].copy()

# Guardar sample del dataset limpio
df_small.shape
os.makedirs("data/processed/exploratory", exist_ok=True)
df_small.to_parquet("data/processed/exploratory/m5_clean_sample.parquet", index=False)

print("Dataset limpio guardado.")

Dataset limpio guardado.


# SECCIÓN 7 — Limitaciones detectadas y necesidad de paralelización
Las pruebas realizadas sobre el dataset completo en formato long evidenciaron un crecimiento sustancial en el número de registros, alcanzando más de 58 millones de filas.<br>
Este volumen de datos implica tiempos de ejecución elevados en operaciones como joins y transformaciones temporales cuando se utilizan herramientas basadas exclusivamente en pandas.<br>
Estas limitaciones motivaron la adopción de Apache Spark para la ejecución del pipeline de transformación, permitiendo paralelizar las operaciones y gestionar eficientemente grandes volúmenes de datos.<br>
Las transformaciones desarrolladas en este notebook constituyen la base lógica que posteriormente será implementada en el pipeline distribuido.